In [3]:
#import + session DB + subgraph storage obj

import pandas as pd

from HOGDB.graph.subgraph import Subgraph          # constructeur Subgraph (API HO-GDB)
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase

# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)

In [ ]:
# run this in neo4j BD
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Person) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Post) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Tag) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Place) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Forum) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Comment) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Organisation) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:TagClass) REQUIRE n.id IS UNIQUE;



In [ ]:
#funct to resolve the problem of ' in the text content of property values

def escape_cypher_string(value):
    if isinstance(value, str):
        return value.replace("'", "\\'")
    return value


In [ ]:
#cache Person nodes via HOGDB
#we will need  those cache to import edges 

PERSON_CSV = "data/person_0_0.csv"

print("== Insert Person nodes ==")

person_cache = {}
person_buffer = []

person_df = pd.read_csv(
    PERSON_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in person_df.iterrows():
    node = Node(
        labels=[Label("Person")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("firstName", str, escape_cypher_string(row["firstName"])),
            Property("lastName", str, escape_cypher_string(row["lastName"])),
            Property("gender", str, row["gender"]),
            Property("birthday", int, int(row["birthday"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"])
        ]
    )

    
    person_cache[int(row["id"])] = node
    

print(f"== Total Person nodes inserted: {count} ==")


In [ ]:
len(person_cache)

In [ ]:
#add person nodes 

import pandas as pd
csv_path = "mini_data/person_0_0.csv"

def insert_person_nodes(csv_path, gs, batch_size=1000):
    print("== Insert Person nodes ==")

    person_cache = {}
    person_buffer = []

    person_df = pd.read_csv(
        csv_path,
        sep="|",
        header=0,
        engine="python",
        quotechar='"',
        escapechar='\\',
        encoding="utf-8"
    )

    count = 0

    for _, row in person_df.iterrows():
        node = Node(
            labels=[Label("Person")],
            properties=[
                Property("id", int, int(row["id"])),
                Property("firstName", str, escape_cypher_string(row["firstName"])),
                Property("lastName", str, escape_cypher_string(row["lastName"])),
                Property("gender", str, row["gender"]),
                Property("birthday", int, int(row["birthday"])),
                Property("creationDate", int, int(row["creationDate"])),
                Property("locationIP", str, row["locationIP"]),
                Property("browserUsed", str, row["browserUsed"])
            ]
        )

        person_cache[int(row["id"])] = node
        person_buffer.append(node)

        if len(person_buffer) == batch_size:
            for n in person_buffer:
                gs.add_node(n)

            count += batch_size
            print(f"  committed {count} Person nodes")
            person_buffer.clear()

    # last batch
    if person_buffer:
        for n in person_buffer:
            gs.add_node(n)

        count += len(person_buffer)

    print(f"== Total Person nodes inserted: {count} ==")

    return person_cache
person_cache = insert_person_nodes(csv_path, gs, batch_size=1000)

In [ ]:
# cache Post nodes via HOGDB
#we will need  those cache to import edges 

POST_CSV = "data/post_0_0.csv"

print("== Insert Post nodes ==")

post_cache = {}
post_buffer = []

post_df = pd.read_csv(
    POST_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in post_df.iterrows():
    node = Node(
        labels=[Label("Post")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("imageFile", str, row["imageFile"]),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("language", str, row["language"]),
            Property("length", int, int(row["length"]))
        ]
    )

    post_cache[int(row["id"])] = node


print(f"== Total Post nodes inserted: {count} ==")


In [ ]:
# add Post nodes via HOGDB
#we will need  those cache to import edges 

POST_CSV = "data/post_0_0.csv"

print("== Insert Post nodes ==")

post_cache = {}
post_buffer = []

post_df = pd.read_csv(
    POST_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in post_df.iterrows():
    node = Node(
        labels=[Label("Post")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("imageFile", str, row["imageFile"]),
            Property("creationDate", int, int(row["creationDate"])),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("language", str, row["language"]),
            Property("length", int, int(row["length"]))
        ]
    )

    post_cache[int(row["id"])] = node
    post_buffer.append(node)

    if len(post_buffer) == NODE_BATCH:
        for n in post_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Post nodes")
        post_buffer.clear()

# last batch
if post_buffer:
    for n in post_buffer:
        gs.add_node(n)

    count += len(post_buffer)

print(f"== Total Post nodes inserted: {count} ==")


In [ ]:
#get comment_cache

# my paths
COMMENT_CSV = "data/comment_0_0.csv"

NODE_BATCH = 1000






# here  A CACHE

comment_cache = {}   # dict key : id , val : Node



# here  buffers

# here  buffers
comment_buffer = []
tag_buffer = []      
edge_buffer = []   
print("== Insert Comment nodes ==")

comment_df = pd.read_csv(
    COMMENT_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
comment_buffer = []

for _, row in comment_df.iterrows():
    node = Node(
        labels=[Label("Comment")],   # _node sera ajouté par HO-GDB
        properties=[
            Property("id", int, int(row["id"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("length", int, int(row["length"]))
        ]
    )

    comment_cache[int(row["id"])] = node
    



print(f"== Total Comment nodes inserted: {count} ==")


In [ ]:
len(comment_cache)

In [ ]:
#add comment nodes via HOGDB

# my paths
COMMENT_CSV = "data/comment_0_0.csv"

NODE_BATCH = 1000






# here  A CACHE

comment_cache = {}   # dict key : id , val : Node



# here  buffers

# here  buffers
comment_buffer = []
tag_buffer = []      
edge_buffer = []   
print("== Insert Comment nodes ==")

comment_df = pd.read_csv(
    COMMENT_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
comment_buffer = []

for _, row in comment_df.iterrows():
    node = Node(
        labels=[Label("Comment")],   # _node sera ajouté par HO-GDB
        properties=[
            Property("id", int, int(row["id"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("length", int, int(row["length"]))
        ]
    )

    comment_cache[int(row["id"])] = node
    comment_buffer.append(node)

    # batch
    if len(comment_buffer) == NODE_BATCH:
        for n in comment_buffer:
            gs.add_node(n)

                     
        count += NODE_BATCH
        print(f"  committed {count} Comment nodes")
        comment_buffer.clear()

# dernier batch
if comment_buffer:
    for n in comment_buffer:
        gs.add_node(n)
    
    count += len(comment_buffer)

print(f"== Total Comment nodes inserted: {count} ==")


In [ ]:
# get tag_cache



# my paths

TAG_CSV = "data/tag_0_0.csv"


NODE_BATCH = 10



# here  A CACHE


tag_cache = {}       # key : id , val Node


# here  buffer

tag_buffer = []      

print("== Insert Tag nodes ==")

tag_df = pd.read_csv(
    TAG_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
tag_buffer = []

for _, row in tag_df.iterrows():
    node = Node(
       labels=[Label("Tag")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tag_cache[int(row["id"])] = node
    
print(f"== Total tag nodes inserted: {len(tag_cache)} ==")


In [ ]:
# add tag nodes via HOGDB



# my paths

TAG_CSV = "data/tag_0_0.csv"


NODE_BATCH = 10



# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)


# here  A CACHE


tag_cache = {}       # key : id , val Node


# here  buffer

tag_buffer = []      

print("== Insert Tag nodes ==")

tag_df = pd.read_csv(
    TAG_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
tag_buffer = []

for _, row in tag_df.iterrows():
    node = Node(
       labels=[Label("Tag")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tag_cache[int(row["id"])] = node
    tag_buffer.append(node)

    # batch
    if len(tag_buffer) == NODE_BATCH:
        for n in tag_buffer:
            gs.add_node(n)

                     
        count += NODE_BATCH
        print(f"  committed {count} tag nodes")
        tag_buffer.clear()

# dernier batch
if tag_buffer:
    for n in tag_buffer:
        gs.add_node(n)
    
    count += len(tag_buffer)

print(f"== Total tag nodes inserted: {count} ==")


In [ ]:
# cache Place nodes via HOGDB

PLACE_CSV = "data/place_0_0.csv"

print("== Insert Place nodes ==")

place_cache = {}
place_buffer = []

place_df = pd.read_csv(
    PLACE_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in place_df.iterrows():
    node = Node(
        labels=[Label("Place")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"])),
            Property("type", str, row["type"])
        ]
    )

    place_cache[int(row["id"])] = node
    

print(f"== Total Place nodes inserted: {count} ==")


In [ ]:
len(place_cache)

In [ ]:
# add Place nodes via HOGDB

PLACE_CSV = "data/place_0_0.csv"

print("== Insert Place nodes ==")

place_cache = {}
place_buffer = []

place_df = pd.read_csv(
    PLACE_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in place_df.iterrows():
    node = Node(
        labels=[Label("Place")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"])),
            Property("type", str, row["type"])
        ]
    )

    place_cache[int(row["id"])] = node
    place_buffer.append(node)

    if len(place_buffer) == NODE_BATCH:
        for n in place_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Place nodes")
        place_buffer.clear()

# last batch
if place_buffer:
    for n in place_buffer:
        gs.add_node(n)

    count += len(place_buffer)

print(f"== Total Place nodes inserted: {count} ==")


In [ ]:
# cache Forum nodes via HOGDB

FORUM_CSV = "data/forum_0_0.csv"



forum_cache = {}
forum_buffer = []

forum_df = pd.read_csv(
    FORUM_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in forum_df.iterrows():
    node = Node(
        labels=[Label("Forum")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("title", str, escape_cypher_string(row["title"])),
            Property("creationDate", int, int(row["creationDate"]))
        ]
    )

    forum_cache[int(row["id"])] = node
    
print(f"== Total forum cache: {count} ==")


In [ ]:
# add Forum nodes via HOGDB

FORUM_CSV = "mini_data/forum_0_0.csv"

print("== Insert Forum nodes ==")

forum_cache = {}
forum_buffer = []

forum_df = pd.read_csv(
    FORUM_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in forum_df.iterrows():
    node = Node(
        labels=[Label("Forum")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("title", str, escape_cypher_string(row["title"])),
            Property("creationDate", int, int(row["creationDate"]))
        ]
    )

    forum_cache[int(row["id"])] = node
    forum_buffer.append(node)

    if len(forum_buffer) == NODE_BATCH:
        for n in forum_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Forum nodes")
        forum_buffer.clear()

# last batch
if forum_buffer:
    for n in forum_buffer:
        gs.add_node(n)

    count += len(forum_buffer)

print(f"== Total Forum nodes inserted: {count} ==")


In [ ]:
# cache Organisation nodes via HOGDB

ORG_CSV = "data/organisation_0_0.csv"



org_cache = {}
org_buffer = []

org_df = pd.read_csv(
    ORG_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in org_df.iterrows():
    node = Node(
        labels=[Label("Organisation")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("type", str, row["type"]),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    org_cache[int(row["id"])] = node
    

print(f"== Total Organisation nodes inserted: {count} ==")


In [ ]:
len(org_cache)

In [ ]:
# add Organisation nodes via HOGDB

ORG_CSV = "data/organisation_0_0.csv"

print("== Insert Organisation nodes ==")

org_cache = {}
org_buffer = []

org_df = pd.read_csv(
    ORG_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in org_df.iterrows():
    node = Node(
        labels=[Label("Organisation")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("type", str, row["type"]),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    org_cache[int(row["id"])] = node
    org_buffer.append(node)

    if len(org_buffer) == NODE_BATCH:
        for n in org_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Organisation nodes")
        org_buffer.clear()

# last batch
if org_buffer:
    for n in org_buffer:
        gs.add_node(n)

    count += len(org_buffer)

print(f"== Total Organisation nodes inserted: {count} ==")


In [ ]:
# cache TagClass nodes via HOGDB

TAGCLASS_CSV = "data/tagclass_0_0.csv"


tagclass_cache = {}
tagclass_buffer = []

tagclass_df = pd.read_csv(
    TAGCLASS_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in tagclass_df.iterrows():
    node = Node(
        labels=[Label("TagClass")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tagclass_cache[int(row["id"])] = node
    
print(f"== Total TagClass nodes inserted: {count} ==")


In [ ]:
# add TagClass nodes via HOGDB

TAGCLASS_CSV = "data/tagclass_0_0.csv"

print("== Insert TagClass nodes ==")

tagclass_cache = {}
tagclass_buffer = []

tagclass_df = pd.read_csv(
    TAGCLASS_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in tagclass_df.iterrows():
    node = Node(
        labels=[Label("TagClass")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tagclass_cache[int(row["id"])] = node
    tagclass_buffer.append(node)

    if len(tagclass_buffer) == NODE_BATCH:
        for n in tagclass_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} TagClass nodes")
        tagclass_buffer.clear()

# last batch
if tagclass_buffer:
    for n in tagclass_buffer:
        gs.add_node(n)

    count += len(tagclass_buffer)

print(f"== Total TagClass nodes inserted: {count} ==")


In [ ]:
print("NOW WE ADD THE EDGES ")

In [ ]:
#add constraint
#run in neo4j
CREATE CONSTRAINT IF NOT EXISTS FOR (e:_edge) REQUIRE e.id IS UNIQUE;

In [ ]:
#add id in the edges csv files 

# genere un nouveau CSV avec colonne id 
import pandas as pd
import os

# dossier

DATA_DIR = "data/edges_sans_id"
DATA_ID_DIR ="data/edges"

# list of edge files + prefix id
edge_files = {
    "comment_hasCreator_person_0_0.csv": "hasCreator",
    "comment_hasTag_tag_0_0.csv": "hasTag",
    "comment_isLocatedIn_place_0_0.csv": "isLocatedIn",
    "comment_replyOf_comment_0_0.csv": "replyOfComment",
    "comment_replyOf_post_0_0.csv": "replyOfPost",
    "forum_containerOf_post_0_0.csv": "containerOf",
    "forum_hasMember_person_0_0.csv": "hasMember",
    "forum_hasModerator_person_0_0.csv": "hasModerator",
    "forum_hasTag_tag_0_0.csv": "forumHasTag",
    "person_hasInterest_tag_0_0.csv": "hasInterest",
    "person_isLocatedIn_place_0_0.csv": "personLocatedIn",
    "person_knows_person_0_0.csv": "knows",
    "person_likes_comment_0_0.csv": "likesComment",
    "person_likes_post_0_0.csv": "likesPost",
    "person_studyAt_organisation_0_0.csv": "studyAt",
    "person_workAt_organisation_0_0.csv": "workAt",
    "post_hasCreator_person_0_0.csv": "postCreator",
    "post_hasTag_tag_0_0.csv": "postHasTag",
    "post_isLocatedIn_place_0_0.csv": "postLocatedIn",
    "organisation_isLocatedIn_place_0_0.csv": "orgLocatedIn",
    "place_isPartOf_place_0_0.csv": "isPartOf",
    "tag_hasType_tagclass_0_0.csv": "hasType",
    "tagclass_isSubclassOf_tagclass_0_0.csv": "subclassOf"
}

print("== Add ID to all edge CSV files ==")

for file_name, prefix in edge_files.items():
    src = os.path.join(DATA_DIR, file_name)
    dst = os.path.join(DATA_ID_DIR, file_name.replace(".csv", "_with_id.csv"))

    print(f"Processing {file_name}...")

    df = pd.read_csv(src, sep="|", engine="python", encoding="utf-8")

    #créer id unique
    df["id"] = [f"{prefix}{i+1}" for i in range(len(df))]

    # mettre id en première colonne
    cols = ["id"] + [c for c in df.columns if c != "id"]
    df = df[cols]

    # sauvegarde
    df.to_csv(dst, sep="|", index=False, encoding="utf-8")

    print(f"   -> Wrote {dst} ({len(df)} rows)")

print("== DONE =====")


In [10]:
#generic func to add all type of edges 

import pandas as pd

EDGE_BATCH = 1000

def insert_edges(csv_path, source_col, target_col,
                 source_cache, target_cache,
                 label_name, extra_props=[]):

    print(f"== Insert {label_name} edges ==")

    edge_df = pd.read_csv(
        csv_path,
        sep="|",
        header=0,
        engine="python",
        encoding="utf-8"
    )

    edge_buffer = []
    count = 0

    for _, row in edge_df.iterrows():
        s_id = int(row[source_col])
        t_id = int(row[target_col])

        # vérifier existence des nodes
        if s_id not in source_cache or t_id not in target_cache:
            continue

        # propriétés de base
        props = [Property("id", str, row["id"])]

        # propriétés supplémentaires
        for prop in extra_props:
            props.append(Property(prop, str, row[prop]))

        e = Edge(
            source_cache[s_id],
            target_cache[t_id],
            Label(label_name),
            props
        )

        edge_buffer.append(e)

        # batch insert
        if len(edge_buffer) == EDGE_BATCH:
            for edge in edge_buffer:
                gs.add_edge(edge)

            count += EDGE_BATCH
            print(f"  committed {count} {label_name}")
            edge_buffer.clear()

    # dernier batch
    if edge_buffer:
        for edge in edge_buffer:
            gs.add_edge(edge)

        count += len(edge_buffer)

    print(f"== Total {label_name}: {count} ==")


In [32]:
# func to get comment HASTAG cache_edge (needed for subgraph creation)

import pandas as pd

EDGE_BATCH = 1000

def insert_ht_cache_edges(csv_path, source_col, target_col,
                 source_cache, target_cache,
                 label_name, extra_props=[]):

    print(f"== Insert {label_name} edges ==")

    edge_df = pd.read_csv(
        csv_path,
        sep="|",
        header=0,
        engine="python",
        encoding="utf-8"
    )
    edge_cache = {}      # Key : (c_id, t_id) , val : Edge
    edge_buffer = []
    count = 0

    for _, row in edge_df.iterrows():
        s_id = int(row[source_col])
        t_id = int(row[target_col])

        # vérifier existence des nodes
        if s_id not in source_cache or t_id not in target_cache:
            continue

        # propriétés de base
        props = [Property("id", str, row["id"])]

        # propriétés supplémentaires
        for prop in extra_props:
            props.append(Property(prop, str, row[prop]))

        e = Edge(
            source_cache[s_id],
            target_cache[t_id],
            Label(label_name),
            props
        )
        edge_cache[s_id, t_id] = e
   

        
    count = len(edge_cache)

    print(f"== Total : {count} ==")
    return edge_cache

In [33]:
def insert_ht_cache_edges(csv_path, source_col, target_col,
                 source_cache, target_cache,
                 label_name, extra_props=[]):

    print(f"== Insert {label_name} edges ==")

    edge_df = pd.read_csv(
        csv_path,
        sep="|",
        header=0,
        encoding="utf-8"
    )

    edge_cache = {}

    for _, row in edge_df.iterrows():

        if pd.isna(row[source_col]) or pd.isna(row[target_col]):
            continue

        s_id = int(row[source_col])
        t_id = int(row[target_col])

        if s_id not in source_cache or t_id not in target_cache:
            continue

        props = [Property("id", str, str(row["id"]))]

        e = Edge(
            source_cache[s_id],
            target_cache[t_id],
            Label(label_name),
            props
        )

        edge_cache[(s_id, t_id)] = e

    print(f"== Total : {len(edge_cache)} ==")
    return edge_cache

In [ ]:
#get comment HASTAG cache_edge (needed for subgraph creation)
edge_cache= insert_ht_cache_edges(
    "data/comment_hasTag_tag_0_0_with_id.csv",
    "Comment.id", "Tag.id",
    comment_cache, tag_cache,
    "HAS_TAG",extra_props=[]
)
len(edge_cache)

In [ ]:
#get forum HASMEMEBER  cache_edge (needed for subgraph creation)
edgefm_cache= insert_ht_cache_edges(
    "data/forum_hasMember_person_0_0_with_id.csv",
    "Forum.id", "Person.id",
    forum_cache, person_cache,
    "HAS_MEMBER",extra_props=["joinDate"]
)


In [ ]:
len (edgefm_cache)

In [ ]:
#forum HASMEMEBER 
insert_edges(
   "mini_data/forum_hasMember_person_0_0_with_id.csv",
    "Forum.id", "Person.id",
    forum_cache, person_cache,
    "HAS_MEMBER",extra_props=["joinDate"]
)
print("done")


In [ ]:
#Comment HAS TAG tag
insert_edges(
    "data/edges/comment_hasTag_tag_0_0_with_id.csv",
    "Comment.id", "Tag.id",
    comment_cache, tag_cache,
    "HAS_TAG"
)
print("done")


In [ ]:
#COMMENT HAS CREATOR PERSON
insert_edges(
    "data/edges/comment_hasCreator_person_0_0_with_id.csv",
    "Comment.id", "Person.id",
    comment_cache, person_cache,
    "HAS_CREATOR"
)
print("done")

In [ ]:
#Comment IS_LOCATED_IN place 
insert_edges(
    "data/edges/comment_isLocatedIn_place_0_0_with_id.csv",
    "Comment.id", "Place.id",
    comment_cache, place_cache,
    "IS_LOCATED_IN"
)
print("done")

In [ ]:
#COMMENT REPLY OF COMMENT
insert_edges(
    "data/edges/comment_replyOf_comment_0_0_with_id.csv",
    "Comment1.id", "Comment2.id",
    comment_cache, comment_cache,
    "REPLY_OF"
)
print("done")

In [ ]:
#COMMENT REPLYOF POST

insert_edges(
    "data/edges/comment_replyOf_post_0_0_with_id.csv",
    "Comment.id", "Post.id",
    comment_cache, post_cache,
    "REPLY_OF"
)
print("done")

In [ ]:
#FORUM CONTAINTEROF POST
insert_edges(
    "data/edges/forum_containerOf_post_0_0_with_id.csv",
    "Forum.id", "Post.id",
    forum_cache, post_cache,
    "CONTAINER_OF"
)


In [ ]:
#HAS_MEMBER
insert_edges(
    "data/edges/forum_hasMember_person_0_0_with_id.csv",
    "Forum.id", "Person.id",
    forum_cache, person_cache,
    "HAS_MEMBER",
    extra_props=["joinDate"]
)


In [ ]:
#HAS_MODERATOR
insert_edges(
    "data/edges/forum_hasModerator_person_0_0_with_id.csv",
    "Forum.id", "Person.id",
    forum_cache, person_cache,
    "HAS_MODERATOR"
)
print("done")

In [ ]:
#Person KNOWS Person
insert_edges(
    "data/edges/person_knows_person_0_0_with_id.csv",
    "Person1.id", "Person2.id",
    person_cache, person_cache,
    "KNOWS",
    extra_props=["creationDate"]
)


In [ ]:
#Person IS_LOCATED_IN place 
insert_edges(
    "data/edges/person_isLocatedIn_place_0_0_with_id.csv",
    "Person.id", "Place.id",
    person_cache, place_cache,
    "IS_LOCATED_IN"
)
print("done")



In [ ]:
len(place_cache)

In [ ]:
# PERSON LIKES COMMENT
insert_edges(
    "data/edges/person_likes_comment_0_0_with_id.csv",
    "Person.id", "Comment.id",
    person_cache, comment_cache,
    "LIKES",
    extra_props=["creationDate"]
)


In [ ]:
#PERSON LIKES POST
insert_edges(
    "data/edges/person_likes_post_0_0_with_id.csv",
    "Person.id", "Post.id",
    person_cache, post_cache,
    "LIKES",
    extra_props=["creationDate"]
)


In [ ]:
#STUDY_ATabs

insert_edges(
    "data/edges/person_studyAt_organisation_0_0_with_id.csv",
    "Person.id", "Organisation.id",
    person_cache, org_cache,
    "STUDY_AT",
    extra_props=["classYear"]
)



In [ ]:
#WORKS AT
insert_edges(
    "data/edges/person_workAt_organisation_0_0_with_id.csv",
    "Person.id", "Organisation.id",
    person_cache, org_cache,
    "WORK_AT",
    extra_props=["workFrom"]
)

In [ ]:
#POST HAS_CREATOR PERSON
insert_edges(
    "data/edges/post_hasCreator_person_0_0_with_id.csv",
    "Post.id", "Person.id",
    post_cache, person_cache,
    "HAS_CREATOR"
)
print ("done")


In [ ]:
#POST HAS_TAG TAG
insert_edges(
    "data/edges/post_hasTag_tag_0_0_with_id.csv",
    "Post.id", "Tag.id",
    post_cache, tag_cache,
    "HAS_TAG"
)


In [ ]:
#POST IS_LOCATED_IN PLACE
insert_edges(
    "data/edges/post_isLocatedIn_place_0_0_with_id.csv",
    "Post.id", "Place.id",
    post_cache, place_cache,
    "IS_LOCATED_IN"
)


In [ ]:
#organisation IS_LOCATED_IN place 
insert_edges(
    "data/edges/organisation_isLocatedIn_place_0_0_with_id.csv",
    "Organisation.id", "Place.id",
    org_cache, place_cache,
    "IS_LOCATED_IN"
)
print("done")




In [ ]:
#PLACE IS_PART_OF PPLACE
insert_edges(
    "data/edges/place_isPartOf_place_0_0_with_id.csv",
    "Place1.id", "Place2.id",
    place_cache, place_cache,
    "IS_PART_OF"
)




In [ ]:
#TAG HAS_TYPE TTAGCLASS
insert_edges(
    "data/edges/tag_hasType_tagclass_0_0_with_id.csv",
    "Tag.id", "TagClass.id",
    tag_cache, tagclass_cache,
    "HAS_TYPE"
)


In [ ]:
# tagclass IS_SUBCLASS_OF tagclass
insert_edges(
    "data/edges/tagclass_isSubclassOf_tagclass_0_0_with_id.csv",
    "TagClass1.id", "TagClass2.id",
    tagclass_cache, tagclass_cache,
    "IS_SUBCLASS_OF"
)

In [ ]:
# Plan B in case the insertin is interrupted
from itertools import islice

edge_cache_copy = dict(islice(edge_cache.items(), 613237, None))


In [ ]:

#add taggedComment Subgraphs

SUBGRAPH_BATCH = 1000

print("===Insert taggedComment subgraphs ")

buffer = []
count = 0

for (s_id, t_id), e in edge_cache_copy.items():
    sg = Subgraph(
        subgraph_nodes=[
            comment_cache[s_id],
            tag_cache[t_id]
        ],
        subgraph_edges=[e],
        labels=[Label("taggedComment")],
        properties=[
            Property("id", str, f"tc_{s_id}_{t_id}")
        ]
    )

    buffer.append(sg)

    if len(buffer) == SUBGRAPH_BATCH:
        for s in buffer:
            gs.add_subgraph(s)
        count += len(buffer)
        buffer.clear()
        print(f"  inserted {count} subgraphs")

# flush last batch
if buffer:
    for s in buffer:
        gs.add_subgraph(s)
    count += len(buffer)

print(f"== DONE: {count} subgraphs inserted ==")





In [ ]:
# add forumMembership Subgraphs

SUBGRAPH_BATCH = 1000

print("=== Insert forumMembership subgraphs ===")

buffer = []
count = 0

# group persons and edges by forum
forum_members = {}

for (f_id, p_id), e in edgefm_cache.items():

    if f_id not in forum_members:
        forum_members[f_id] = {
            "persons": [],
            "edges": []
        }

    forum_members[f_id]["persons"].append(person_cache[p_id])
    forum_members[f_id]["edges"].append(e)


# create subgraphs
for f_id, data in forum_members.items():

    forum_node = forum_cache[f_id]
    persons = data["persons"]
    edges = data["edges"]

    sg = Subgraph(
        subgraph_nodes=[forum_node] + persons,
        subgraph_edges=edges,
        labels=[Label("forumMembership")],
        properties=[
            Property("id", str, f"fm_{f_id}"),
            Property("nbMember", int, len(persons))
        ]
    )

    buffer.append(sg)

    if len(buffer) == SUBGRAPH_BATCH:
        for s in buffer:
            gs.add_subgraph(s)

        count += len(buffer)
        buffer.clear()
        print(f"  inserted {count} subgraphs")


# flush last batch
if buffer:
    for s in buffer:
        gs.add_subgraph(s)
    count += len(buffer)

print(f"== DONE: {count} forumMembership subgraphs inserted ==")

In [ ]:
# CLOSE
gs.close_connection()